# Sales Forecasting - Exploratory Data Analysis (EDA)

This notebook performs initial exploration of the UCI Online Retail dataset.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys

# Add src to path
sys.path.insert(0, str(Path('..').resolve()))

from src.data_processing import load_excel_data, get_data_info, filter_valid_transactions, add_sales_column

# Set style
plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

# Screenshot settings
SCREENSHOTS_DIR = Path('../reports/screenshots')
SCREENSHOTS_DIR.mkdir(parents=True, exist_ok=True)

def save_screenshot(fig, name):
    """Save figure to screenshots directory."""
    filepath = SCREENSHOTS_DIR / f'{name}.png'
    fig.savefig(filepath, dpi=150, bbox_inches='tight')
    print(f'Saved: {filepath}')

## 1. Load Raw Data

In [2]:
# Load the dataset
raw_data_path = Path('../data/raw/Online Retail.xlsx')
df_raw = load_excel_data(raw_data_path)

print(f"Dataset shape: {df_raw.shape}")
print(f"\nColumn names: {list(df_raw.columns)}")
df_raw.head()

Dataset shape: (541909, 8)

Column names: ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


## 2. Data Overview

In [3]:
# Get dataset info
info = get_data_info(df_raw)

print("Dataset Information:")
print(f"- Rows: {info['shape'][0]:,}")
print(f"- Columns: {info['shape'][1]}")
print(f"- Memory usage: {info['memory_usage_mb']:.2f} MB")
print(f"\nMissing values:")
for col, count in info['missing_values'].items():
    if count > 0:
        print(f"  - {col}: {count:,} ({count/info['shape'][0]*100:.1f}%)")

Dataset Information:
- Rows: 541,909
- Columns: 8
- Memory usage: 104.99 MB

Missing values:
  - Description: 1,454 (0.3%)
  - CustomerID: 135,080 (24.9%)


In [4]:
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[us]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 40.0+ MB


In [5]:
df_raw.describe()

,Quantity,InvoiceDate,UnitPrice,CustomerID
count,541909.000000,541909,541909.000000,406829.000000
mean,9.552250,2011-07-04 13:34:57.156386,4.611114,15287.690570
min,-80995.000000,2010-12-01 08:26:00,-11062.060000,12346.000000
25%,1.000000,2011-03-28 11:34:00,1.250000,13953.000000
50%,3.000000,2011-07-19 17:17:00,2.080000,15152.000000
75%,10.000000,2011-10-19 11:27:00,4.130000,16791.000000
max,80995.000000,2011-12-09 12:50:00,38970.000000,18287.000000
std,218.081158,NaN,96.759853,1713.600303


## 3. Data Cleaning

In [6]:
# Filter valid transactions
df_clean = filter_valid_transactions(df_raw)
df_clean = add_sales_column(df_clean)

print(f"Original shape: {df_raw.shape}")
print(f"Cleaned shape: {df_clean.shape}")
print(f"Removed: {df_raw.shape[0] - df_clean.shape[0]:,} rows ({(df_raw.shape[0] - df_clean.shape[0])/df_raw.shape[0]*100:.1f}%)")

Original shape: (541909, 8)
Cleaned shape: (397884, 9)
Removed: 144,025 rows (26.6%)


## 4. Basic Statistics

In [7]:
print("Sales Statistics:")
print(f"- Total Sales: £{df_clean['Sales'].sum():,.2f}")
print(f"- Average Transaction: £{df_clean['Sales'].mean():,.2f}")
print(f"- Median Transaction: £{df_clean['Sales'].median():,.2f}")
print(f"- Max Transaction: £{df_clean['Sales'].max():,.2f}")
print(f"\nUnique Values:")
print(f"- Customers: {df_clean['CustomerID'].nunique():,}")
print(f"- Products: {df_clean['StockCode'].nunique():,}")
print(f"- Invoices: {df_clean['InvoiceNo'].nunique():,}")
print(f"- Countries: {df_clean['Country'].nunique()}")

Sales Statistics:
- Total Sales: £8,911,407.90
- Average Transaction: £22.40
- Median Transaction: £11.80
- Max Transaction: £168,469.60

Unique Values:
- Customers: 4,338
- Products: 3,665
- Invoices: 18,532
- Countries: 37


## 5. Initial Visualizations

In [ ]:
# Sales distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Filter for reasonable sales range (remove extreme outliers)
sales_filtered = df_clean['Sales'][(df_clean['Sales'] > 0) & (df_clean['Sales'] < 500)]

# Histogram of sales
axes[0].hist(sales_filtered, bins=50, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Sales (£)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Transaction Sales (< £500)')

# Top 10 countries by sales
country_sales = df_clean.groupby('Country')['Sales'].sum().sort_values(ascending=False).head(10)
country_sales.plot(kind='bar', ax=axes[1], edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Country')
axes[1].set_ylabel('Total Sales (£)')
axes[1].set_title('Top 10 Countries by Sales')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
save_screenshot(fig, '01_sales_distribution')
plt.show()

## 6. Save Cleaned Data

In [ ]:
# Save cleaned data to CSV
output_path = Path('../data/processed/cleaned_retail_data.csv')
df_clean.to_csv(output_path, index=False)
print(f"Saved cleaned data to: {output_path}")
print(f"Shape: {df_clean.shape}")